# 08 - RAG Pipeline Test

Week 8 end-to-end validation for retrieval, generator, deterministic explainer, and the pipeline wrappers.

- Group A: pure RAG Q&A
- Group B: RAG-augmented prediction
- If no valid LLM credentials are available, Group A is skipped gracefully and Group B still runs.

In [1]:
import json
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import shap

REPO_ROOT = Path('../').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.load_usaid import load_clean, time_split
from src.data.load_unctad import load_lookup
from src.features.features import build_features
from src.rag.pipeline import rag_augmented_prediction, rag_query
from src.rag.retriever import retrieve

MODEL_BUNDLE = joblib.load(REPO_ROOT / 'models' / 'usaid_model.pkl')
MODEL = MODEL_BUNDLE['model']
FEATURE_NAMES = MODEL_BUNDLE['feature_names']
UNCTAD_LOOKUP = load_lookup()

print(f'Repo root: {REPO_ROOT}')
print(f'Model loaded: {type(MODEL).__name__}')
print(f'LLM provider: {os.getenv("LLM_PROVIDER", "groq")}')

Repo root: E:\freight-cost-rag
Model loaded: LGBMRegressor
LLM provider: groq


In [2]:
def has_llm_credentials() -> bool:
    provider = os.getenv('LLM_PROVIDER', 'groq').strip().lower()
    if provider == 'groq':
        return bool(os.getenv('GROQ_API_KEY'))
    if provider == 'openai':
        return bool(os.getenv('OPENAI_API_KEY'))
    return False

def record_result(results, group, name, status, payload):
    results.append({
        'group': group,
        'name': name,
        'status': status,
        'payload': payload,
    })

def display_payload(title, payload):
    print(f'\n=== {title} ===')
    if isinstance(payload, dict):
        print(json.dumps(payload, indent=2, default=str)[:3000])
    else:
        print(payload)

results = []

## Group A - Pure RAG Q&A

In [3]:
group_a_cases = [
    ('A1', 'What does DDP mean?'),
    ('A2', 'What are current ocean freight rates?'),
    ('A3', 'Explain the difference between CIF and FOB'),
    ('A4', 'What is the capital of France?'),
    ('A5', 'Which HS chapter covers pharmaceutical products?'),
]

if not has_llm_credentials():
    note = 'Skipped Group A: no valid API key found for the active provider.'
    print(note)
    for case_id, query in group_a_cases:
        record_result(results, 'A', case_id, 'SKIP', {'query': query, 'note': note})
else:
    for case_id, query in group_a_cases:
        retrieved = retrieve(query, top_k=3)
        response = rag_query(query)
        status = 'PASS'
        if case_id == 'A4' and response['answer'].strip() != "I don't have enough information to answer that.":
            status = 'FAIL'
        payload = {
            'query': query,
            'retrieved_chunks': retrieved,
            'response': response,
        }
        display_payload(case_id, payload)
        record_result(results, 'A', case_id, status, payload)

Skipped Group A: no valid API key found for the active provider.


## Group B - RAG-Augmented Prediction

In [4]:
usaid = load_clean()
train_df, test_df = time_split(usaid)
X_train = build_features(train_df, unctad_lookup=UNCTAD_LOOKUP)
X_test = build_features(test_df, unctad_lookup=UNCTAD_LOOKUP)
explainer = shap.TreeExplainer(MODEL)
shap_values = explainer.shap_values(X_test)

mode_targets = {
    'Air': 'Nigeria',
    'Ocean': 'South Africa',
    'Truck': 'Ethiopia',
}

selected_rows = {}
for mode, dest in mode_targets.items():
    match = test_df[(test_df['mode'] == mode) & (test_df['dest_country'] == dest)]
    if match.empty:
        match = test_df[test_df['mode'] == mode]
    selected_rows[mode] = match.index[0]

group_b_cases = [
    ('B1', 'Air', 'Nigeria'),
    ('B2', 'Ocean', 'South Africa'),
    ('B3', 'Truck', 'Ethiopia'),
]

E:\freight-cost-rag\src\data\load_usaid.py:80: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["delivery_date"] = pd.to_datetime(df["Scheduled Delivery Date"], dayfirst=True, errors="coerce")


In [5]:
for case_id, mode, destination in group_b_cases:
    row_idx = selected_rows[mode]
    feature_row = X_test.loc[row_idx, FEATURE_NAMES]
    prediction_usd = float(np.expm1(MODEL.predict(feature_row.to_frame().T)[0]))
    explanation = rag_augmented_prediction(
        prediction_usd=prediction_usd,
        shap_values=shap_values[list(X_test.index).index(row_idx)],
        feature_names=FEATURE_NAMES,
        feature_values=feature_row.to_dict(),
        shipment_mode=mode,
        destination_country=destination,
    )
    status = 'PASS'
    if case_id == 'B1' and 'destination-tier' not in explanation['explanation'].lower():
        status = 'FAIL'
    if case_id == 'B3' and explanation['market_context']:
        top_text = explanation['market_context'][0]['text'].lower()
        if mode.lower() == 'truck' and 'road' not in top_text and 'truck' not in top_text:
            status = 'FAIL'
    payload = {
        'mode': mode,
        'destination': destination,
        'prediction_usd': prediction_usd,
        'market_context': explanation['market_context'],
        'explanation': explanation['explanation'],
    }
    display_payload(case_id, payload)
    record_result(results, 'B', case_id, status, payload)

record_result(results, 'B', 'B4', 'PASS' if any('destination-tier' in r['payload'].get('explanation', '').lower() for r in results if r['group'] == 'B') else 'FAIL', {'note': 'UNCTAD destination-tier framing check'})
record_result(results, 'B', 'B5', 'PASS', {'note': 'Mode-relevance checked within individual Group B cases'})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


=== B1 ===
{
  "mode": "Air",
  "destination": "Nigeria",
  "prediction_usd": 4859.764286680527,
  "market_context": [
    {
      "text": "## Freight Rate Benchmarks for African Corridors\n\n**Important provenance note:** Lane-specific spot rates for Africa-bound air freight (e.g. Europe\u2192Nairobi, US\u2192Nairobi) are not publicly available from the major indices (Freightos FAX, Xeneta Air) without subscription access. The figures below are structural order-of-magnitude ranges drawn from logistics provider published quotes and industry reports \u2014 they are directional context, not verified spot rates. Do not cite them as benchmark data in reports.\n\n**Air freight (structural ranges, not verified spot rates):**\n- Published logistics provider quotes for US\u2192Kenya: USD 7.80\u201320.00/kg (wide range reflecting service tier differences, not market rates)\n- Global air cargo composite spot rate: USD 2.86\u20133.76/kg (March\u2013April 2026, WorldACD via Bertling report; this 

## Summary

In [6]:
summary_df = pd.DataFrame(results)
summary_df[['group', 'name', 'status']]


,group,name,status
0,A,A1,SKIP
1,A,A2,SKIP
2,A,A3,SKIP
3,A,A4,SKIP
4,A,A5,SKIP
5,B,B1,FAIL
6,B,B2,PASS
7,B,B3,PASS
8,B,B4,PASS
9,B,B5,PASS


### Markdown Summary

The cell below produces a compact markdown report with pass/fail/skip counts and per-case notes.

In [7]:
counts = summary_df['status'].value_counts().to_dict()
lines = [
    '# Week 8 RAG Pipeline Summary',
    '',
    f"- PASS: {counts.get('PASS', 0)}",
    f"- FAIL: {counts.get('FAIL', 0)}",
    f"- SKIP: {counts.get('SKIP', 0)}",
    '',
    '## Case Results',
]
for row in results:
    note = row['payload'].get('note', '') if isinstance(row['payload'], dict) else ''
    lines.append(f"- {row['name']} [{row['status']}] {note}".rstrip())
summary_markdown = '\n'.join(lines)
print(summary_markdown)

# Week 8 RAG Pipeline Summary

- PASS: 4
- FAIL: 1
- SKIP: 5

## Case Results
- A1 [SKIP] Skipped Group A: no valid API key found for the active provider.
- A2 [SKIP] Skipped Group A: no valid API key found for the active provider.
- A3 [SKIP] Skipped Group A: no valid API key found for the active provider.
- A4 [SKIP] Skipped Group A: no valid API key found for the active provider.
- A5 [SKIP] Skipped Group A: no valid API key found for the active provider.
- B1 [FAIL]
- B2 [PASS]
- B3 [PASS]
- B4 [PASS] UNCTAD destination-tier framing check
- B5 [PASS] Mode-relevance checked within individual Group B cases
